<a href="https://colab.research.google.com/github/Dev1ze32/llma3.2_3b_model_test/blob/main/content/02-model_test/jupyter_notebooks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
import torch
import json
import time
from typing import TypedDict, Annotated, Literal
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from huggingface_hub import login

login()

In [20]:
# ─────────────────────────────────────────
# LOAD MODEL ONCE, SHARE ACROSS ALL NODES
# ─────────────────────────────────────────

model_id = "meta-llama/Llama-3.2-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

print("Model ready.")

Loading tokenizer...
Loading model...


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Model ready.


In [21]:
# ─────────────────────────────────────────
# STATE DEFINITION
# ─────────────────────────────────────────

from typing import TypedDict

class ChatState(TypedDict):
    messages: list          # plain list, no add_messages reducer
    route: str
    confidence: float
    response: str

In [22]:
# ─────────────────────────────────────────
# SYSTEM PROMPTS
# ─────────────────────────────────────────

CLASSIFIER_SYSTEM = """You are a message classifier for University of Cabuyao helpdesk.

Your only job is to classify the student's LATEST message into one category.
Use the conversation history to understand context of follow-up messages.

CATEGORIES:

FORMS
- Anything about university forms and registrar processes
- Examples:
  "How do I drop a subject?"
  "What form do I need to shift programs?"
  "Paano mag apply ng LOA?"
  "I want to get my clearance"
  "How do I correct my name in my records?"
  "Anong form para sa load adjustment?"
  "Where do I submit my completion form?"

GENERAL
- Anything about university information, policies, internship, data privacy rights
- Examples:
  "What programs does UC offer?"
  "How many OJT hours for BSIT?"
  "What are my rights under the Data Privacy Act?"
  "Where is the university located?"
  "What is the vision of Pamantasan ng Cabuyao?"
  "Anong requirements sa internship?"
  "What does the Data Protection Department do?"

HUMAN
- Anything requiring personal records or direct staff action
- Examples:
  "What is my grade in Math?"
  "Can you check my enrollment status?"
  "I want to file a complaint"
  "Check if my form was already submitted"
  "What is my remaining balance?"
  "I need to talk to someone"

IMPORTANT RULES:
- If the message is a follow-up to a previous topic, classify based on that topic
- If genuinely unclear after reading history, classify as HUMAN
- Never classify personal record lookups as anything other than HUMAN

Respond in this EXACT format, no other text:
{
  "route": "FORMS" or "GENERAL" or "HUMAN",
  "confidence": 0.0 to 1.0,
  "reason": "one sentence explanation"
}"""


FORMS_SYSTEM = """You are the Registrar helpdesk assistant for University of Cabuyao (Pamantasan ng Cabuyao).

You help students understand what forms they need and how to process them.

AVAILABLE FORMS YOU KNOW ABOUT:
- PNC:OUR-FO-01 | Request for Student Load Adjustment
  → Submit to OUR, needs adviser signature

- PNC:OUR-FO-02 | Application for Shifting of Curricular Program
  → Submit to OUR, needs department acceptance

- PNC:OUR-FO-03 | Dropping of Courses and Withdrawal Form
  → Submit to OUR before the deadline, needs adviser approval

- PNC:OUR-FO-06 | Request for Leave of Absence
  → Submit to OUR, needs parent/guardian consent if minor

- PNC:OUR-FO-11 | Request for Correction of Student Personal Data
  → Submit to OUR with PSA Birth Certificate or Marriage Contract

- PNC:OUR-FO-19 | Terminal Clearance for Graduating Students
  → For graduating students, multiple office signatures required

- PNC:OUR-FO-21 | Completion Form
  → For incomplete grades, submit to OUR

- PNC:OUR-FO-27 | Addition and Cancellation of Courses Form
  → Submit during enrollment adjustment period

- PNC:OUR-FO-28 | Semestral Clearance Slip
  → Required every semester, multiple office sign-offs

- PNC:MISD-FO-28 | Student ID Application Form
  → Submit to MISD office

RULES:
- Only answer based on what you know from the forms above
- If a student asks about something you are not sure of,
  direct them to the OUR office directly
- Never make up form codes or requirements
- Keep answers clear and step by step
- If student needs personal record confirmation, remind them
  you cannot access records and to visit OUR

Office of the University Registrar (OUR)
University of Cabuyao, Katapatan Mutual Homes,
Brgy. Banay-banay, City of Cabuyao, Laguna 4025"""


GENERAL_SYSTEM = """You are the general information assistant for University of Cabuyao (Pamantasan ng Cabuyao).

You answer questions about the university using only the information below.

UNIVERSITY INFORMATION:

Location: Katapatan Mutual Homes, Brgy. Banay-banay, City of Cabuyao, Laguna 4025

Vision: A premier educational institution of higher learning in Region IV,
developing globally-competitive and value-laden professionals and leaders
instrumental to community development and nation building.

Mission: As an institution of higher learning, PnC is committed to equip
individuals with knowledge, skills and values that will enable them to achieve
their professional goals and provide leadership and service for national development.

Core Values: Personal Dignity, Nurturing Community, Commitment to Excellence

COLLEGES:
- College of Arts and Sciences
- College of Health Allied and Sciences
- College of Education
- College of Business Accountancy and Administration
- College of Computing Studies
- College of Engineering
- Graduate School
- Senior High School

INTERNSHIP HOURS PER PROGRAM:
- BS Psychology: 450 hours (industrial, educational, clinical)
- BS Computer Science: 300 hours
- BS Information Technology: 500 hours
- BS Computer Engineering: 240 hours
- BS Electronics Engineering: 240 hours
- BS Industrial Engineering: 240 hours
- BS Education (Elementary/Secondary): 360 hours
- BS Accountancy: 400 hours
- BS Business Administration: 600 hours
- BS Nursing: 2,703 hours

INTERNSHIP GRADING:
- 15% Weekly Journal Entry
- 60% Student Intern Performance Evaluation
- 25% Internship Portfolio

DATA PRIVACY:
- Data Protection Officer Email: dpd@pnc.edu.ph
- Student Rights: right to be informed, access, object,
  erase/block, rectify, portability, file complaint, damages
- Complaints: fill out Data Privacy Complaint Form or
  email dpd@pnc.edu.ph

RULES:
- Only answer using the information above
- If not covered, say you are unsure and suggest
  which office to contact
- Never make up information
- Keep answers concise and helpful"""


HUMAN_SYSTEM = """You are a helpdesk assistant for University of Cabuyao.

The student is asking something that requires personal records
or direct staff assistance which this chatbot cannot provide.

Your job is to:
1. Acknowledge their concern warmly
2. Explain clearly you cannot access personal information
3. Direct them to the correct office

OFFICE DIRECTORY:
- Grades / Academic Records / Forms
  → Office of the University Registrar (OUR)

- Internship / OJT specific concerns
  → Placement, Alumni, and Linkages Department (PALD)

- Data Privacy / Personal Data Requests / Complaints
  → Data Protection Department (DPD)
  → Email: dpd@pnc.edu.ph

- Financial / Fees / Balance
  → Finance Department

- Student Complaints / Disciplinary
  → Office of Student Affairs

- General / Unclear
  → Main University Office
    Katapatan Mutual Homes, Brgy. Banay-banay,
    City of Cabuyao, Laguna 4025

Keep your response warm, short, and helpful.
Never pretend you can look up their records."""

In [23]:
# ─────────────────────────────────────────
# HELPER: RUN INFERENCE
# ─────────────────────────────────────────

def sanitize_messages(messages: list) -> list:
    """
    Ensure every message only has role and content keys
    Strips any LangGraph metadata that gets injected
    """
    clean = []
    for msg in messages:
        # Handle LangGraph message objects
        if hasattr(msg, "role") and hasattr(msg, "content"):
            clean.append({
                "role": msg.role,
                "content": msg.content
            })
        # Handle plain dicts
        elif isinstance(msg, dict):
            if "role" in msg and "content" in msg:
                clean.append({
                    "role": msg["role"],
                    "content": msg["content"]
                })
        # Skip anything malformed
        else:
            continue
    return clean


def run_inference(
    system_prompt: str,
    messages: list,
    max_new_tokens: int = 512,
    deterministic: bool = False
) -> str:

    # Sanitize first
    clean_messages = sanitize_messages(messages)

    # Build formatted list with system prompt
    formatted = [{"role": "system", "content": system_prompt}]
    formatted.extend(clean_messages)

    # Debug: uncomment to inspect what goes into pipeline
    # print(f"[DEBUG] Formatted messages: {formatted}")

    outputs = pipe(
        formatted,
        max_new_tokens=max_new_tokens,
        do_sample=not deterministic,
        temperature=0.3 if not deterministic else 1.0,
        repetition_penalty=1.1
    )

    return outputs[0]["generated_text"][-1]["content"]


def parse_classifier_output(raw: str) -> dict:
    try:
        start = raw.find("{")
        end = raw.rfind("}") + 1
        json_str = raw[start:end]
        result = json.loads(json_str)
        result["parse_success"] = True
        return result
    except:
        return {
            "route": "HUMAN",  # safe fallback
            "confidence": 0.0,
            "reason": "parse failed, defaulting to human",
            "parse_success": False,
            "raw": raw
        }

In [24]:
# ─────────────────────────────────────────
# LANGGRAPH NODES
# ─────────────────────────────────────────

def classifier_node(state: ChatState) -> ChatState:
    print("\n[CLASSIFIER] Running...")

    raw = run_inference(
        system_prompt=CLASSIFIER_SYSTEM,
        messages=state["messages"],
        max_new_tokens=100,
        deterministic=True
    )

    result = parse_classifier_output(raw)

    print(f"[CLASSIFIER] Route: {result['route']} | "
          f"Confidence: {result['confidence']} | "
          f"Reason: {result['reason']}")

    if result["confidence"] < 0.65:
        print("[CLASSIFIER] Low confidence, routing to HUMAN")
        return {
            **state,
            "route": "HUMAN",
            "confidence": result["confidence"]
        }

    return {
        **state,
        "route": result["route"],
        "confidence": result["confidence"]
    }


def forms_node(state: ChatState) -> ChatState:
    print("\n[FORMS NODE] Generating response...")

    response = run_inference(
        system_prompt=FORMS_SYSTEM,
        messages=state["messages"],
        max_new_tokens=512
    )

    # Manually append assistant response
    updated_messages = state["messages"] + [
        {"role": "assistant", "content": response}
    ]

    return {
        **state,
        "response": response,
        "messages": updated_messages
    }


def general_node(state: ChatState) -> ChatState:
    print("\n[GENERAL NODE] Generating response...")

    response = run_inference(
        system_prompt=GENERAL_SYSTEM,
        messages=state["messages"],
        max_new_tokens=512
    )

    updated_messages = state["messages"] + [
        {"role": "assistant", "content": response}
    ]

    return {
        **state,
        "response": response,
        "messages": updated_messages
    }


def human_node(state: ChatState) -> ChatState:
    print("\n[HUMAN NODE] Escalating...")

    response = run_inference(
        system_prompt=HUMAN_SYSTEM,
        messages=state["messages"],
        max_new_tokens=256
    )

    updated_messages = state["messages"] + [
        {"role": "assistant", "content": response}
    ]

    return {
        **state,
        "response": response,
        "messages": updated_messages
    }

# ─────────────────────────────────────────
# ROUTING FUNCTION
# called after classifier to pick next node
# ─────────────────────────────────────────

def route_decision(state: ChatState) -> Literal["forms", "general", "human"]:
    route = state.get("route", "HUMAN")

    mapping = {
        "FORMS":   "forms",
        "GENERAL": "general",
        "HUMAN":   "human"
    }

    return mapping.get(route, "human")

In [25]:
# Rebuild graph with same structure
builder = StateGraph(ChatState)

builder.add_node("classifier", classifier_node)
builder.add_node("forms",      forms_node)
builder.add_node("general",    general_node)
builder.add_node("human",      human_node)

builder.set_entry_point("classifier")

builder.add_conditional_edges(
    "classifier",
    route_decision,
    {
        "forms":   "forms",
        "general": "general",
        "human":   "human"
    }
)

builder.add_edge("forms",   END)
builder.add_edge("general", END)
builder.add_edge("human",   END)

graph = builder.compile()

print("Graph rebuilt successfully.")

Graph rebuilt successfully.


In [26]:
# ─────────────────────────────────────────
# FIX CHAT CLASS
# History is now just plain dicts always
# ─────────────────────────────────────────

class UCHelpdesk:
    def __init__(self):
        self.history = []   # always plain dicts only

    def chat(self, user_message: str) -> str:

        # Append user message as plain dict
        self.history.append({
            "role": "user",
            "content": user_message
        })

        # Build state with plain list
        initial_state: ChatState = {
            "messages": self.history.copy(),  # copy to avoid mutation
            "route": "",
            "confidence": 0.0,
            "response": ""
        }

        result = graph.invoke(initial_state)

        # Pull back only clean messages from result
        self.history = sanitize_messages(result["messages"])

        return result["response"]

    def reset(self):
        self.history = []
        print("Conversation reset.")

    def show_history(self):
        print("\n── Conversation History ──")
        for msg in self.history:
            role = msg["role"].upper()
            print(f"\n[{role}]: {msg['content']}")
        print("\n─────────────────────────")

In [ ]:
# Rerun interactive
helpdesk = UCHelpdesk()

print("UC Helpdesk ready. Type 'quit' to exit, 'reset' to clear history.\n")

while True:
    user_input = input("You: ").strip()

    if user_input.lower() == "quit":
        break
    elif user_input.lower() == "reset":
        helpdesk.reset()
        continue
    elif not user_input:
        continue

    response = helpdesk.chat(user_input)
    print(f"\nBot: {response}\n")

UC Helpdesk ready. Type 'quit' to exit, 'reset' to clear history.

You: how to enroll


The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'repetition_penalty', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[CLASSIFIER] Running...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[CLASSIFIER] Route: GENERAL | Confidence: 0.8 | Reason: Enrollment process typically falls under general university information

[GENERAL NODE] Generating response...

Bot: To enroll at the University of Cabuyao (Pamantasan ng Cabuyao), please follow these steps:

1. Visit our website: [insert website URL]
2. Click on "Admissions" and then "Apply Online"
3. Fill out the online application form with required details such as personal information, academic background, and course preferences.
4. Submit your application and supporting documents (e.g., high school diploma, transcripts, etc.) via email or through our online portal.
5. Pay the required enrollment fees and examination fees (if applicable).
6. Wait for a response from our Admissions Office regarding your application status.

For more detailed information, I recommend contacting our Admissions Office directly:

[Insert contact information]

Please note that specific requirements may vary depending on the program and course you're

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[CLASSIFIER] Running...


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[CLASSIFIER] Route: FORMS | Confidence: 1.0 | Reason: Dropping a subject typically involves submitting a Form 1429, which can be obtained from the Registrar's office.

[FORMS NODE] Generating response...

Bot: If you need to drop a subject, you'll need to submit the following form:

**PNC:OUR-FO-03 | Dropping of Courses and Withdrawal Form**

This form can be obtained from our office or downloaded from our website. Please fill it out accurately and completely.

Here are the steps to follow:

1. Download and print the form (PNC:OUR-FO-03) from our website or obtain one from our office.
2. Fill out the form with your personal details, course details, and reason for dropping the subject.
3. Get your course advisor's signature on the form, indicating their approval of your request to drop the subject.
4. Submit the completed form to our office (Office of the University Registrar, OUR) before the deadline specified in the Academic Calendar.
5. Note that there might be additional requirement